* [Velog: RAG 기본 시스템 구축하기](https://velog.io/@dlgkdis801/Day-03-Plus2.-%EA%B8%B0%EB%B3%B8-RAG-%EC%8B%9C%EC%8A%A4%ED%85%9C-%EA%B5%AC%EC%B6%95%ED%95%98%EA%B8%B0)

In [1]:
# 라이브러리 다 있는 지 확인
from setup_env import ensure_packages
ensure_packages()

# PDF 파일 로드
from dotenv import load_dotenv
import os
load_dotenv()
from loaders import load_all_pdfs
GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")

pages = load_all_pdfs(GITHUB_TOKEN)
print("총 페이지:", len(pages))

# 청크로 분할 및 벡터로 저장
from splitter import split_pages
from embeddings import NVIDIAEmbeddingsCustom
from langchain_chroma import Chroma

docs = split_pages(pages, chunk_size=1000, chunk_overlap=100)
print("총 청크:", len(docs))

NVIDIA_API_KEY = os.getenv("NVIDIA_BUILD_KEY")
embeddings = NVIDIAEmbeddingsCustom(NVIDIA_API_KEY)
vectorstore = Chroma.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever()
print("retriever 준비 완료")

# LLM 선언 및 RAG_Chain 선언
from llm import get_llm
from chain import build_rag_chain, stream_response

llm = get_llm(NVIDIA_API_KEY)
rag_chain = build_rag_chain(retriever, llm)

모든 패키지 준비 완료
처리 중...
2023년도 사전정보공표 리스트_230701(게시용).pdf
처리 중...
2023년도 사전정보공표 항목 및 담당부서(게시용).pdf
처리 중...
2024년도 사전정보공표 항목 및 담당부서(게시용).pdf
총 페이지: 12
총 청크: 40
retriever 준비 완료
Chain 선언 완료


In [2]:
user_prompt = "첨부한 파일에 대해서 설명해줘. 혹시 7월에는 공표가 안되었나?"
stream_response(rag_chain, user_prompt)

첨부된 파일은 재무·회계, 정보화·보안, 기술·교육, 계약·구매 등 다양한 분야의 정보 공개 항목과 공표 주기를 정리한 목록입니다. 7월에는 보유도서 목록, 보안점검 결과, 500만 원 이상 수의계약 현황, 단위 정부과제 목록 등 여러 항목이 공표되는 것으로 확인됩니다.
